<a href="https://colab.research.google.com/github/mondweep/ruvllm-assurance-demo/blob/main/notebooks/ruvllm_paired_replication.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RuVLLM: trust but verify

Re-run rUv's paired adapter test on your own Colab GPU, then mark the same answers three ways.

1. **Act 1, side by side (~10 min):** one task per tool family, base model vs `coding-4b` adapter.
2. **Act 2, full replication (~1–2 h):** all 256 held-out tasks, both models, with rUv's own worker and scorer.
3. **Act 3, fair marking (seconds):** strict, format-tolerant and fair markers over your saved answers.

**GPU:** choose *Runtime → Change runtime type → L4 or A100*. A T4 will not work: the worker requires native BF16.

Upstream code is pinned to `ruvnet/ruvllm@890a5d7` (Apache 2.0). Nothing here executes the model's tool calls.

## 0. Check the GPU

In [2]:
import subprocess
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                     capture_output=True, text=True).stdout.strip()
print(gpu or 'No GPU attached')
assert gpu, 'Attach a GPU: Runtime → Change runtime type'
assert not gpu.startswith('Tesla T4'), 'T4 has no native BF16. Switch to an L4 or A100.'

NVIDIA L4, 23034 MiB


## 1. Save outputs to Google Drive
Colab sessions can disconnect; runs written to Drive survive.

In [3]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
RUNS = Path('/content/drive/MyDrive/ruvllm-demo/runs')
RUNS.mkdir(parents=True, exist_ok=True)
print('Runs will be saved under', RUNS)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Runs will be saved under /content/drive/MyDrive/ruvllm-demo/runs


## 2. Get the code (pinned)

In [4]:
%cd /content
!test -d ruvllm || git clone -q https://github.com/ruvnet/ruvllm.git
!git -C ruvllm checkout -q 890a5d72c9550bb53df7e57b2b4503fa20dd080f && git -C ruvllm log -1 --format='ruvllm @ %h %s'
!test -d ruvllm-assurance-demo || git clone -q https://github.com/mondweep/ruvllm-assurance-demo.git
!git -C ruvllm-assurance-demo pull -q

/content
ruvllm @ 890a5d7 Add scrollytelling tour and tutorial site; refocus README on measured capabilities (#6)


## 3. Install the pinned software

The worker refuses to run on any other versions. This replaces Colab's own torch, which is why we
haven't imported torch yet. If an import fails later, use *Runtime → Restart session* and carry on from step 4.

In [5]:
!pip -q install torch==2.11.0 transformers==5.12.1 peft==0.21.2 safetensors==0.7.0 huggingface_hub
!python -c "import torch, transformers, peft; print('torch', torch.__version__, '| CUDA', torch.version.cuda, '| native BF16', torch.cuda.is_bf16_supported(including_emulation=False)); print('transformers', transformers.__version__, '| peft', peft.__version__)"

torch 2.11.0+cu130 | CUDA 13.0 | native BF16 True
transformers 5.12.1 | peft 0.21.2


In [6]:
!pip uninstall -y torchao

## 4. Download the pinned base model (~9 GB, a few minutes)

In [7]:
import json
from huggingface_hub import snapshot_download
MODEL_DIR = '/content/models/Qwen3.5-4B-851bf6e'
M = Path('/content/ruvllm/experiments/model-improvement')
assets = json.loads((M / 'model-assets.json').read_text())
snapshot_download(repo_id='Qwen/Qwen3.5-4B', revision='851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a',
                  local_dir=MODEL_DIR, token=False,
                  allow_patterns=[*assets['asset_files'], 'model*.safetensors'])
print('Model ready at', MODEL_DIR)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

Model ready at /content/models/Qwen3.5-4B-851bf6e


## Act 1: side by side

One task from each tool family on the held-out panel. Watch the *grep* row: the base model puts
the folder into the `glob` filter instead of `path`. And the *test* row: the base adds a harmless
`timeout`, which the strict marker counts as wrong.

In [8]:
import sys, torch
sys.path.insert(0, '/content/ruvllm-assurance-demo/scoring')
from fair_rescore import grade
from transformers import AutoTokenizer, Qwen3_5ForConditionalGeneration
from peft import PeftModel

cases = json.loads((M / 'frozen/v4/cases.json').read_text())
picks, seen = [], set()
for c in cases:
    fam = c['id'].split('-tool-')[1].rsplit('-', 1)[0] if c['kind'] == 'tool' else 'math'
    if fam not in seen:
        seen.add(fam); picks.append((fam, c))

tok = AutoTokenizer.from_pretrained(MODEL_DIR, local_files_only=True)
base = Qwen3_5ForConditionalGeneration.from_pretrained(MODEL_DIR, local_files_only=True, dtype=torch.bfloat16,
                                                       device_map='cuda', attn_implementation='sdpa')
model = PeftModel.from_pretrained(base, '/content/ruvllm/weights/coding-4b', is_trainable=False,
                                  low_cpu_mem_usage=False, autocast_adapter_dtype=True).eval()

def answer(case, use_adapter):
    ids = tok.apply_chat_template(case['messages'], tools=case['tools'], tokenize=True,
                                  add_generation_prompt=True, enable_thinking=False, return_dict=False)
    x = torch.tensor([ids], device='cuda')
    run = lambda: model.generate(input_ids=x, attention_mask=torch.ones_like(x), max_new_tokens=512,
                                 do_sample=False, eos_token_id=[248044, 248046])
    with torch.inference_mode():
        if use_adapter:
            out = run()
        else:
            with model.disable_adapter():
                out = run()
    gen = out[0, len(ids):].tolist()
    finish = 'stop' if gen and gen[-1] in (248044, 248046) else 'length'
    return dict(text=tok.decode(gen, skip_special_tokens=False), finish_reason=finish)

def verdict(case, row):
    return ' '.join(f"{m}:{'✓' if grade(case, row, m) else '✗'}" for m in ('strict', 'format', 'fair'))

for fam, case in picks:
    b, a = answer(case, False), answer(case, True)
    print(f"\n=== {fam}: {case['messages'][-1]['content'][:110]}")
    print('BASE   ', verdict(case, b), '|', ' '.join(b['text'].split())[:260])
    print('ADAPTER', verdict(case, a), '|', ' '.join(a['text'].split())[:260])

del model, base; import gc; gc.collect(); torch.cuda.empty_cache()

[ERROR] `loss` is part of Qwen3_5CausalLMOutputWithPast.__init__'s signature, but not documented. Make sure to add it to the docstring of the function in /usr/local/lib/python3.13/dist-packages/transformers/models/qwen3_5/modeling_qwen3_5.py.
[ERROR] `logits` is part of Qwen3_5CausalLMOutputWithPast.__init__'s signature, but not documented. Make sure to add it to the docstring of the function in /usr/local/lib/python3.13/dist-packages/transformers/models/qwen3_5/modeling_qwen3_5.py.


[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.



=== read: Before changing anything, look at every line of /workspace/heldout_v4/read/heron_00/src/heron_00.py.
BASE    strict:✓ format:✓ fair:✓ | <tool_call> <function=Read> <parameter=file_path> /workspace/heldout_v4/read/heron_00/src/heron_00.py </parameter> </function> </tool_call><|im_end|>
ADAPTER strict:✓ format:✓ fair:✓ | <tool_call> <function=Read> <parameter=file_path> /workspace/heldout_v4/read/heron_00/src/heron_00.py </parameter> </function> </tool_call><|im_end|>


[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


... (11 outputs left out)

[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.



=== test: Run this approved command exactly as written and nothing besides it: python -m pytest /workspace/heldout_v4/te
BASE    strict:✓ format:✓ fair:✓ | <tool_call> <function=Bash> <parameter=command> python -m pytest /workspace/heldout_v4/test/heron_00/tests/test_heron_00.py -x </parameter> </function> </tool_call><|im_end|>
ADAPTER strict:✓ format:✓ fair:✓ | <tool_call> <function=Bash> <parameter=command> python -m pytest /workspace/heldout_v4/test/heron_00/tests/test_heron_00.py -x </parameter> </function> </tool_call><|im_end|>


[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.



=== math: Held-out arithmetic 00: what is 29 * (5 + 2) - 37?
BASE    strict:✓ format:✓ fair:✓ | To compute the value of the expression $29 \times (5 + 2) - 37$, we follow the order of operations (PEMDAS/BODMAS): 1. **Parentheses**: Calculate the value inside the parentheses first. $$5 + 2 = 7$$ Now the expression becomes: $$29 \times 7 - 37$$ 2. **Multip
ADAPTER strict:✓ format:✓ fair:✓ | To compute the value of the expression $29 \times (5 + 2) - 37$, we follow the order of operations (PEMDAS/BODMAS): 1. **Parentheses**: Calculate the value inside the parentheses first. $$5 + 2 = 7$$ Now the expression becomes: $$29 \times 7 - 37$$ 2. **Multip


## Act 2: full paired replication

rUv's own one-command runner: fresh processes for each arm, hash checks on the model and adapter,
greedy decoding, then rUv's strict scorer. Expect roughly 1–2 hours on an L4 (not yet timed).

Your GPU differs from rUv's RTX 3090, so a few answers may differ. Treat this as an independent replication.

In [ ]:
import datetime, shutil
stamp = datetime.datetime.now().strftime('%Y%m%d-%H%M%S')
RUN_V4 = RUNS / f'v4-coding-4b-{stamp}'
if RUN_V4.exists():
    shutil.rmtree(RUN_V4)
!cd /content/ruvllm/experiments/model-improvement && python3 run-pair.py \
  --bundle /content/ruvllm/experiments/model-improvement/frozen/v4 --model-dir {MODEL_DIR} --adapter-dir /content/ruvllm/weights/coding-4b \
  --out {RUN_V4} --deadline-seconds 5400
panels = json.loads((RUN_V4 / 'scored/report.json').read_text())['panels']
for kind in ('tool', 'math'):
    p = panels[kind]
    print(f"{kind}: base {p['base_correct']} → adapter {p['adapter_correct']} of {p['count']} "
          f"({p['gains']} gains, {p['regressions']} regressions, p = {p['exact_mcnemar_p']:.2g})")

[ERROR] `loss` is part of Qwen3_5CausalLMOutputWithPast.__init__'s signature, but not documented. Make sure to add it to the docstring of the function in /usr/local/lib/python3.13/dist-packages/transformers/models/qwen3_5/modeling_qwen3_5.py.
[ERROR] `logits` is part of Qwen3_5CausalLMOutputWithPast.__init__'s signature, but not documented. Make sure to add it to the docstring of the function in /usr/local/lib/python3.13/dist-packages/transformers/models/qwen3_5/modeling_qwen3_5.py.
[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d
Loading weights: 100% 723/723 [00:02<00:00, 309.74it/s]
{"arm": "base", "completed": 1, "total": 256}
{"arm": "base", "completed": 2, "total": 256}
{"arm": "base", "completed": 3, "total": 256}
{"arm": "base", "completed": 4, "total": 256}
{"arm"

Optional: the rejected `coding-4b-r2` adapter on the same tasks (another ~1 h).

In [ ]:
RUN_V5 = RUNS / f'v5-coding-4b-r2-{stamp}'
!cd /content/ruvllm/experiments/model-improvement && python3 run-pair.py \
  --bundle frozen/v5 --model-dir {MODEL_DIR} --adapter-dir /content/ruvllm/weights/coding-4b-r2 \
  --out {RUN_V5} --deadline-seconds 5400 2>&1 | tail -n 3

## Act 3: mark the same answers three ways

- **strict:** rUv's marker, reimplemented; it must reproduce `report.json` or the script stops.
- **format:** also accepts `True`/`False` and ignores trailing whitespace.
- **fair:** also accepts harmless optional settings (`Bash.timeout`, `Bash.description`, `Grep.-n`).

In [ ]:
from IPython.display import Markdown, display
pairs = [f'--pair "my v4: base vs coding-4b" {M}/frozen/v4 {RUN_V4}']
if 'RUN_V5' in globals() and (RUN_V5 / 'scored/report.json').exists():
    pairs.append(f'--pair "my v5: base vs coding-4b-r2" {M}/frozen/v5 {RUN_V5}')
!python3 -I /content/ruvllm-assurance-demo/scoring/fair_rescore.py --upstream {M} --out {RUNS}/fair-{stamp} {' '.join(pairs)} > /dev/null
display(Markdown((RUNS / f'fair-{stamp}' / 'fair-rescore.md').read_text()))

## What to compare against

rUv's published runs, marked the same three ways, are in
[`results/published/fair-rescore.md`](https://github.com/mondweep/ruvllm-assurance-demo/blob/main/results/published/fair-rescore.md).
On the held-out panel, strict marking shows +27 for `coding-4b`. Fair marking shows +24, all in one tool family (grep).